# 02 — Byte Pair Encoding: Subword Tokenization

**Lecture goal:** understand *why* whole-word vocabularies break, build a tiny Byte Pair Encoding (BPE) algorithm by hand to see exactly how it works, then switch to the production-grade implementation (`tiktoken`) that real GPT models use.

## Recap of the problem from notebook 01

Our `SimpleTokenizerV2` vocabulary only contained whole words that appeared in `the-verdict.txt`. Anything else became `<|unk|>` — a single, generic "I don't know this" token that throws away all information about the word. A typo, a rare technical term, a name, a word in another language: all collapse to the same `<|unk|>` ID. That's a lot of lost signal, and it also means our vocabulary would need to be *enormous* to cover realistic amounts of English, let alone code or other languages.

**Byte Pair Encoding (BPE)** solves this by tokenizing at the *subword* level instead of the whole-word level. Common words stay as single tokens (`"the"` → one token), but rare or unseen words get broken into smaller, still-meaningful pieces (`"tokenization"` → maybe `"token"` + `"ization"`). Critically, since the smallest possible pieces are individual characters (or bytes), **every possible string can be represented — there is no more `<|unk|>`.**

## Three levels of tokenization, and why the middle one wins

Before building BPE it helps to see the whole design space. There are really only three choices for what counts as a token, and they sit on a spectrum.

**Word-level** — what notebook 01 built. Split the sentence into words; each word is a token. Three things go wrong:

1. **Out-of-vocabulary words.** Anything unseen is unrepresentable, which is why we needed `<|unk|>`.
2. **Root words are lost.** `"boy"` and `"boys"` obviously share a root, but they get two unrelated integer IDs. Nothing in the representation says they're related — that similarity is thrown away before the model ever sees the text.
3. **The vocabulary is enormous.** English has somewhere between 600,000 and a million words; a vocabulary that size would be unwieldy, and it still wouldn't cover names, typos, code, or other languages.

**Character-level** — the opposite extreme. `"my hobby"` becomes `m`, `y`, `h`, `o`, `b`, `b`, `y`. This fixes the first and third problems beautifully: there are only a couple of hundred characters in a language, so the vocabulary is tiny, and no string is ever unrepresentable. But it makes the second problem worse and adds a new one:

- **Meaning is gone entirely.** Breaking a word down to letters removes exactly the thing that made it a word. Nothing connects `"boy"` to `"boys"` any more than it connects either to `"box"`.
- **Sequences get much longer.** `"hobby"` was one token; now it's five. Across a large corpus that multiplies the amount of computation for the same amount of text — and, as we'll see in notebook 05, attention cost grows with the square of sequence length.

**Subword-level** — the compromise, and what GPT uses. Tokens can be whole words, subwords, *or* single characters, governed by two rules:

- Frequently used words are **not** split; they stay single tokens.
- Rare words **are** split into smaller meaningful pieces.

Apply that to `"boy"` and `"boys"`: `"boy"` is common, so it stays whole; `"boys"` is rarer, so it becomes `"boy"` + `"s"`. Now the two share a token, and the shared root is visible in the representation — which is what lets the model learn that `"token"`, `"tokens"` and `"tokenizing"` are related. Meanwhile the vocabulary stays a manageable ~50,000, and because single characters remain available as tokens, nothing is ever unrepresentable.

Best of both ends of the spectrum: a modest vocabulary, no `<|unk|>`, and root-word structure preserved. The remaining question is *how* to decide which pieces make good tokens — and that's the algorithm we'll build next.

## How BPE actually works

BPE wasn't invented for language models at all. It dates from 1994 and was a **data compression** algorithm: find the most frequent pair of adjacent bytes in the data, replace every occurrence with a single new symbol, and repeat. Each round shortens the data by trading a repeated pair for one symbol. Applied to text instead of bytes, that exact "merge the most frequent pair" loop turns out to implement the two subword rules above for free — frequent sequences get merged into single tokens precisely *because* they're frequent, while rare ones are left in pieces. Nobody has to decide by hand which words are common.

The algorithm that *builds* a BPE vocabulary (this happens once, ahead of time, on a huge training corpus) is surprisingly simple:

1. Start with a base vocabulary of individual characters (or bytes) — every single character seen in the training text is its own token.
2. Look at the training text as sequences of these character-tokens, and count how often every *pair* of adjacent tokens occurs.
3. Take the single most frequent pair, and merge it into one new token. Add that new token to the vocabulary.
4. Repeat steps 2–3 thousands of times. Each round, the vocabulary gains one new (usually longer) token.

The result: extremely common substrings (like `"th"`, then later `"the"`) get merged into single tokens early, because they're frequent. Rare substrings never get merged and stay as small pieces or individual characters.

Let's implement a tiny version of this ourselves on a toy example, so the mechanism is concrete before we use a real, pretrained BPE tokenizer.

In [1]:
from collections import Counter

# A tiny toy corpus. In real BPE this would be billions of characters; here, a handful of words
# repeated so that some pairs are clearly more frequent than others.
toy_words = ["low", "low", "low", "lowest", "newer", "newer", "wider"]

# Step 1: represent each word as a list of individual characters (its starting "tokens"),
# with a special end-of-word marker "_" so the algorithm can tell "er" at the end of a word
# apart from "er" in the middle of one.
corpus = [list(word) + ["_"] for word in toy_words]
for word_tokens in corpus:
    print(word_tokens)

['l', 'o', 'w', '_']
['l', 'o', 'w', '_']
['l', 'o', 'w', '_']
['l', 'o', 'w', 'e', 's', 't', '_']
['n', 'e', 'w', 'e', 'r', '_']
['n', 'e', 'w', 'e', 'r', '_']
['w', 'i', 'd', 'e', 'r', '_']


In [2]:
def get_pair_counts(corpus):
    """Count how often each adjacent pair of tokens occurs, across all words."""
    pair_counts = Counter()
    for word_tokens in corpus:
        for i in range(len(word_tokens) - 1):
            pair = (word_tokens[i], word_tokens[i + 1])
            pair_counts[pair] += 1
    return pair_counts


def merge_pair(pair, corpus):
    """Replace every adjacent occurrence of `pair` with a single merged token."""
    merged_token = pair[0] + pair[1]
    new_corpus = []
    for word_tokens in corpus:
        new_word_tokens = []
        i = 0
        while i < len(word_tokens):
            if (
                i < len(word_tokens) - 1
                and word_tokens[i] == pair[0]
                and word_tokens[i + 1] == pair[1]
            ):
                new_word_tokens.append(merged_token)
                i += 2
            else:
                new_word_tokens.append(word_tokens[i])
                i += 1
        new_corpus.append(new_word_tokens)
    return new_corpus

In [3]:
# Run a handful of merge rounds, printing what gets merged each time.
num_merges = 6
merges = []

for step in range(num_merges):
    pair_counts = get_pair_counts(corpus)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    corpus = merge_pair(best_pair, corpus)
    merges.append(best_pair)
    print(f"Merge {step + 1}: {best_pair} -> '{best_pair[0] + best_pair[1]}'  "
          f"(occurred {pair_counts[best_pair]} times)")

print()
print("Words after merging:")
for word_tokens in corpus:
    print(word_tokens)

Merge 1: ('l', 'o') -> 'lo'  (occurred 4 times)
Merge 2: ('lo', 'w') -> 'low'  (occurred 4 times)
Merge 3: ('low', '_') -> 'low_'  (occurred 3 times)
Merge 4: ('e', 'r') -> 'er'  (occurred 3 times)
Merge 5: ('er', '_') -> 'er_'  (occurred 3 times)
Merge 6: ('n', 'e') -> 'ne'  (occurred 2 times)

Words after merging:
['low_']
['low_']
['low_']
['low', 'e', 's', 't', '_']
['ne', 'w', 'er_']
['ne', 'w', 'er_']
['w', 'i', 'd', 'er_']


Watch what happened: `("l", "o")` merged first because it was the most frequent adjacent pair (`"low"` appears three times, plus once inside `"lowest"` — four occurrences total). A few rounds later `("e", "r")` merged too, driven by `"newer"` (twice) and `"wider"` (once). By the end, `"low"` (padded with the end-of-word marker `"_"`) collapsed into a single token `low_`, while `"wider"` — which shares fewer frequent pairs with the rest of the toy corpus — stayed mostly unmerged as separate characters plus the merged `er_` piece.

This is exactly the mechanism GPT-2's tokenizer uses, just run for ~50,000 merge steps over a huge corpus of internet text instead of 6 steps over 7 toy words. The result is a fixed vocabulary of about 50,000 tokens, along with the ordered list of merge rules, that we can reuse on *any* input text — including words the algorithm never saw, since it can always fall back to spelling a word out character by character.

We are **not** going to retrain BPE ourselves on a huge corpus (that's a multi-hour job even on fast hardware, and OpenAI already did it for GPT-2). Instead, we'll load their exact, pretrained BPE tokenizer using the `tiktoken` library.

## Using `tiktoken`: a production BPE tokenizer

[`tiktoken`](https://github.com/openai/tiktoken) is OpenAI's fast, open-source implementation of BPE, shipping the exact vocabulary and merge rules used by GPT-2, GPT-3.5, and GPT-4-family models. We already added it as a project dependency (`uv add tiktoken`), so we can just import it.

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(type(tokenizer))

<class 'tiktoken.core.Encoding'>


In [5]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

# allowed_special tells tiktoken that "<|endoftext|>" is a real special token here,
# not just literal text to be broken into subwords.
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [6]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


Round trip works, as before. Now the important test: what happens with `"someunknownPlace"` — a word that is almost certainly not a single token in GPT-2's vocabulary? Let's decode the tokens one at a time to see how it got split up.

In [7]:
for token_id in integers:
    print(token_id, "->", repr(tokenizer.decode([token_id])))

15496 -> 'Hello'
11 -> ','
466 -> ' do'
345 -> ' you'
588 -> ' like'
8887 -> ' tea'
30 -> '?'
220 -> ' '
50256 -> '<|endoftext|>'
554 -> ' In'
262 -> ' the'
4252 -> ' sun'
18250 -> 'lit'
8812 -> ' terr'
2114 -> 'aces'
286 -> ' of'
617 -> ' some'
34680 -> 'unknown'
27271 -> 'Place'
13 -> '.'


There it is: `"someunknownPlace"` got broken into several smaller subword pieces (and GPT-2's BPE operates on raw bytes, so unusual casing/boundaries show up as separate pieces too) — no `<|unk|>` in sight. **Every** string, no matter how strange, can be encoded, because in the worst case BPE can always fall back to individual bytes.

This is why every modern LLM uses subword tokenization instead of whole-word tokenization: it gets the efficiency of common words being single tokens, without ever hitting a wall on unfamiliar text.

One more thing worth knowing: the vocabulary size isn't fixed across the GPT family. GPT-2's `gpt2` encoding has 50,257 tokens; the encodings used by GPT-3.5 and GPT-4 (`cl100k_base` and later) are roughly twice that. Later models were given larger vocabularies — more merges means longer common sequences collapse into single tokens, so the same text costs fewer tokens to represent. You can check any of them yourself with `tiktoken.get_encoding(name).n_vocab`.

## Comparing to notebook 01's word-level tokenizer

Let's encode the full short story with `tiktoken` and compare the resulting token count to our word-level `tokenize()` function from notebook 01.

In [8]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

bpe_token_ids = tokenizer.encode(raw_text)
print("Number of BPE tokens:  ", len(bpe_token_ids))
print("First 20 token IDs:    ", bpe_token_ids[:20])
print("Decoded back:          ", tokenizer.decode(bpe_token_ids[:20]))

Number of BPE tokens:   5145
First 20 token IDs:     [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]
Decoded back:           I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--


## Recap

- Tokenization has three levels: **word** (huge vocabulary, `<|unk|>` fallback, root words lost), **character** (tiny vocabulary, no unknowns, but meaning destroyed and sequences several times longer), and **subword** (the compromise GPT uses).
- Subword tokenization keeps frequent words whole and splits rare ones, so `"boy"` and `"boys"` share a token and the shared root survives into the representation.
- **BPE** builds a vocabulary bottom-up: start from individual characters/bytes, repeatedly merge the most frequent adjacent pair. Common substrings become single tokens; rare ones stay as smaller pieces.
- Because the base units are bytes, BPE can represent *any* string — there is no more `<|unk|>`.
- `tiktoken` gives us OpenAI's exact, pretrained GPT-2 BPE tokenizer (vocabulary size 50,257) ready to use via `.encode()` / `.decode()`.

### What's next

We now have a reliable way to turn any text into a sequence of integer token IDs. In notebook 03, we'll take a long sequence of these IDs and figure out how to chop it into the fixed-size, overlapping (input, target) chunks that the model is actually trained on — and load them efficiently in batches using PyTorch's `Dataset` and `DataLoader`.